### RAG Pipelines - Data ingestion to Vector DB Pipeline

In [90]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [91]:
### Read all the PDF's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✕ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Classification Model Evaluation_ Case Study Solutions.pdf
  ✓ Loaded 1 pages

Processing: Regression Diagnostics_ Patient Length of Stay.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [92]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-14T18:48:49+00:00', 'title': 'Classification Model Diagnostics: Case Study Solutions', 'moddate': '2026-10-14T18:48:49+00:00', 'source': '..\\data\\pdf\\Classification Model Evaluation_ Case Study Solutions.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Classification Model Evaluation_ Case Study Solutions.pdf', 'file_type': 'pdf'}, page_content='DIAGNOSTIC MATRIX 1 · CLASSIFICATION SOLUTIONS\nClassification Model Evaluation: Case Study Solutions\nBenchmarked Case Study: An automated email ﬁltering model was evaluated on a veriﬁed test set of 100 emails.\nThe goal is separating Spam (Positive Class) from Legitimate Mail / Ham (Negative Class) with minimal false\nalarms and minimal leaks.\n1. The Confusion Matrix (N = 100)\nGround Truth\nPredicted Condition\nActual Totals\nPredicted Spam (Positive) Predicted Ham (Negative)\nActual Spam (Positive) True Positive (TP) = 4

In [93]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split: {len(documents)} documents into {len(split_docs)} chunks")
    
    # show example of a chunk
    if split_docs:
        print(f"Example chunk content:\n{split_docs[0].page_content[:200]}...")  # Show first 500 characters of the first chunk
        print(f"Example chunk metadata:\n{split_docs[0].metadata}")
        
    return split_docs

In [94]:
chunks = split_documents(all_pdf_documents)
chunks

split: 2 documents into 6 chunks
Example chunk content:
DIAGNOSTIC MATRIX 1 · CLASSIFICATION SOLUTIONS
Classification Model Evaluation: Case Study Solutions
Benchmarked Case Study: An automated email ﬁltering model was evaluated on a veriﬁed test set of 10...
Example chunk metadata:
{'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-14T18:48:49+00:00', 'title': 'Classification Model Diagnostics: Case Study Solutions', 'moddate': '2026-10-14T18:48:49+00:00', 'source': '..\\data\\pdf\\Classification Model Evaluation_ Case Study Solutions.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Classification Model Evaluation_ Case Study Solutions.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-14T18:48:49+00:00', 'title': 'Classification Model Diagnostics: Case Study Solutions', 'moddate': '2026-10-14T18:48:49+00:00', 'source': '..\\data\\pdf\\Classification Model Evaluation_ Case Study Solutions.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Classification Model Evaluation_ Case Study Solutions.pdf', 'file_type': 'pdf'}, page_content='DIAGNOSTIC MATRIX 1 · CLASSIFICATION SOLUTIONS\nClassification Model Evaluation: Case Study Solutions\nBenchmarked Case Study: An automated email ﬁltering model was evaluated on a veriﬁed test set of 100 emails.\nThe goal is separating Spam (Positive Class) from Legitimate Mail / Ham (Negative Class) with minimal false\nalarms and minimal leaks.\n1. The Confusion Matrix (N = 100)\nGround Truth\nPredicted Condition\nActual Totals\nPredicted Spam (Positive) Predicted Ham (Negative)\nActual Spam (Positive) True Positive (TP) = 4

### Embeding And VectorStoreDB 

In [95]:
import os
import uuid
from typing import Any, Dict, List, Tuple

import chromadb
import numpy as np
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [96]:
class EmbeddingManager:
    """Handle document embeddings generation using Sentence Transformers."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager 
        
        Args:
            model_name: HuggingFace model name for  sentence embeddings.
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the Sentence Transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, documents: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of documents.
        
        Args:
            documents: List of document strings to embed.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        
        try:
            embeddings = self.model.encode(documents, convert_to_numpy=True)
            print(f"Generated embeddings for {len(documents)} documents.")
            print(f"Generated Embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
        
## initialize the EmbeddingManager and generate embeddings for the chunks

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1474.14it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\hp_pa\AppData\Local\Temp\ipykernel_18152\3100781073.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


## Vector Store

In [97]:
class VectorStore:
    """A simple vector store using ChromaDB for storing and retrieving document embeddings."""
    
    def __init__(self, collection_name: str = "document_embeddings", persist_directory: str = "../data/vector_store"):
        """
        Initialize the VectorStore
        
        Args:
            collection_name: Name of the ChromaDB collection to use.
            persist_directory: Directory for persisting the vector store data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            os.mkdir(self.persist_directory)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF documment Embedings for RAG"}
            )
            
            print(f"Vector store initialized. collection: {self.collection_name}")
            print(f"Existing documents in the collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise  
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)  # Ensure it's a dictionary
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for ChromaDB
        
        # Add to Collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore = VectorStore()
vectorstore
            

Error initializing vector store: [WinError 183] Cannot create a file when that file already exists: '../data/vector_store'


FileExistsError: [WinError 183] Cannot create a file when that file already exists: '../data/vector_store'

In [ ]:
chunks

[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-14T18:48:49+00:00', 'title': 'Classification Model Diagnostics: Case Study Solutions', 'moddate': '2026-10-14T18:48:49+00:00', 'source': '..\\data\\pdf\\Classification Model Evaluation_ Case Study Solutions.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Classification Model Evaluation_ Case Study Solutions.pdf', 'file_type': 'pdf'}, page_content='DIAGNOSTIC MATRIX 1 · CLASSIFICATION SOLUTIONS\nClassification Model Evaluation: Case Study Solutions\nBenchmarked Case Study: An automated email ﬁltering model was evaluated on a veriﬁed test set of 100 emails.\nThe goal is separating Spam (Positive Class) from Legitimate Mail / Ham (Negative Class) with minimal false\nalarms and minimal leaks.\n1. The Confusion Matrix (N = 100)\nGround Truth\nPredicted Condition\nActual Totals\nPredicted Spam (Positive) Predicted Ham (Negative)\nActual Spam (Positive) True Positive (TP) = 4

In [ ]:
### Convert text to Embeddings and store in vector store
texts = [doc.page_content for doc in chunks]

## Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

## store the embeddings and documents in the vector store
vectorstore.add_documents(chunks, embeddings)

Generated embeddings for 6 documents.
Generated Embeddings with shape: (6, 384)
Adding 6 documents to the vector store...
Successfully added 6 documents to the vector store.
Total documents in the collection after addition: 6


### Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: Any, embedding_manager: Any):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        try:
            # Generate query embedding and extract the first item [0]
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]

            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            # ChromaDB returns nested lists inside a dictionary, so we check and unpack [0]
            if results.get('documents') and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


In [113]:
# 1. Initialize your retriever
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)

# 2. Run the query with an explicit threshold of 0.0 or less to make sure it doesn't filter out results
results = rag_retriever.retrieve(
    query="Classification Model Diagnostics", 
    top_k=5, 
    score_threshold=-1.0  # Force it to show everything regardless of distance score
)

# 3. Pretty print the output to match the dictionary layout in your screenshot
import pprint

pprint.pprint(results)


Retrieving documents for query: 'Classification Model Diagnostics'
Top K: 5, Score threshold: -1.0
Generated embeddings for 1 documents.
Generated Embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
[{'content': 'DIAGNOSTIC MATRIX 1 · CLASSIFICATION SOLUTIONS\n'
             'Classification Model Evaluation: Case Study Solutions\n'
             'Benchmarked Case Study: An automated email ﬁltering model was '
             'evaluated on a veriﬁed test set of 100 emails.\n'
             'The goal is separating Spam (Positive Class) from Legitimate '
             'Mail / Ham (Negative Class) with minimal false\n'
             'alarms and minimal leaks.\n'
             '1. The Confusion Matrix (N = 100)\n'
             'Ground Truth\n'
             'Predicted Condition\n'
             'Actual Totals\n'
             'Predicted Spam (Positive) Predicted Ham (Negative)\n'
             'Actual Spam (Positive) True Positive (TP) = 40Correctly Filtered '
             'Spam Fal

### Integration VectorDB Context pipeline with LLM output

In [120]:
### Simple RAG Pipeline with Groq LLM
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise RuntimeError("GROQ_API_KEY was not found in the .env file")

llm = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024,
)

## Simple RAG Function: retrieve context and generate response
def rag_simple(query, retriever, llm, top_k=3, score_threshold=-1.0):
    print("Starting retrieval...")

    results = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)

    print(f"Retrieved {len(results)} document(s)")

    context = "\n".join(
        doc["content"]
        for doc in results
        if doc.get("content")
    )

    if not context:
        return "No relevant context to answer the question."

    prompt = f"""Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:"""

    print("Calling Groq...")

    response = llm.invoke(prompt)

    print("Groq response received")

    return response.content

In [123]:
from pprint import pprint

answer = rag_simple(
    "What is Classification Model Diagnostics?",
    rag_retriever,
    llm,
)

pprint(
    {
        "Question": "What is classification",
        "Answer": answer,
    },
    sort_dicts=False,
    width=100,
)

Starting retrieval...
Retrieving documents for query: 'What is Classification Model Diagnostics?'
Top K: 3, Score threshold: -1.0
Generated embeddings for 1 documents.
Generated Embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Retrieved 3 document(s)
Calling Groq...
Groq response received
{'Question': 'What is classification',
 'Answer': '**Classification Model Diagnostics** is the systematic examination of how well a '
           'classification model is performing by breaking down its predictions into a confusion '
           'matrix and then computing a set of quantitative metrics that describe different '
           'aspects of that performance.  \n'
           '\n'
           'In practice, diagnostics answer questions such as:\n'
           '\n'
           '| Diagnostic Question | Metric (derived from the confusion matrix) | What it tells you '
           '|\n'
           '|---------------------|--------------------------------------------|------------------

### Enhanced RAG Pipline Features

In [127]:
# --- Enhanced RAG Pipeline with Groq LLM and Metadata Filtering ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = """Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Classification Model Diagnostics?", rag_retriever, llm, top_k=3, min_score=-1.0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'What is Classification Model Diagnostics?'
Top K: 3, Score threshold: -1.0
Generated embeddings for 1 documents.
Generated Embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: Classification Model Diagnostics are the quantitative assessments derived from a model’s confusion matrix that evaluate how well the classifier performs. They include metrics such as accuracy, precision, recall (sensitivity), and F‑score (e.g., F1), each interpreting the proportion of correct predictions, the rate of true positives among predicted positives, the coverage of actual positives, and the balance between precision and recall. Together they reveal strengths and weaknesses—e.g., false‑alarm (FP) and leak (FN) rates—guiding decisions about model suitability and needed adjustments.
Sources: [{'source': 'Classification Model Evaluation_ Case Study Solutions.pdf', 'page': 0, 'score': -0.30690133571624756, 'preview': 'DIAGNOSTIC MATRIX 1 · CLASSIFICA

In [128]:
# --- Advanced RAG Pipleine: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history
        
    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            
        # Streaming answer simulation
        prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
        if stream:
            print("Streaming answer:")
            for i in range(0, len(prompt), 80):
                print(prompt[i:i+80], end='', flush=True)
                time.sleep(0.05)
            print()
            
        response = self.llm.invoke([prompt.format(context=context, question=question)])
        answer = response.content
        
        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer
        
        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content
            
        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })
        
        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generated embeddings for 1 documents.
Generated Embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:


Question: what is attention is all you need

Answer:

Final Answer: “Attention Is All You Need” is a 2017 research paper that introduced the Transformer model—a neural network architecture that relies solely on self‑attention mechanisms (without recurrent or convolutional layers) to process sequences, achieving state‑of‑the‑art performance on tasks like machine translation.
Summary: “Attention Is All You Need” (2017) introduced the Transformer, a neural network architecture that processes sequences using only self‑attention mechanisms, eliminating the need for recurrent or convolutional layers. This design achieved state‑of‑the‑art results on tasks such as machine translation.
Histor